# Logistic Regression Exercise Solutions

---

Solutions to the exercises of `notebooks/R/02_EAIF_Logistic_Regression.ipynb`. The notebook is self-contained: the first code cells rebuild the data, the stratified split, the undersampled training set and the two models exactly as in the main notebook, so every exercise can be run on its own. For each exercise you find the task, one code cell, and the result with its interpretation.

Numbers quoted in the result cells come from a local run of this notebook (R 4.3); Colab may differ in the last digit. R draws a different random sample for the split and the undersampling than the Python twin does, so the two notebooks differ slightly in the numbers, not in the conclusions.


In [ ]:
# Install and load the packages we need (binary builds from the Posit Package Manager)
options(repos = c(CRAN = "https://packagemanager.posit.co/cran/__linux__/jammy/latest"))
required_packages <- c("readr", "tidyr", "ggplot2", "gridExtra", "caret", "pROC", "dplyr")
new_packages <- required_packages[!(required_packages %in% installed.packages()[, "Package"])]
if (length(new_packages) > 0) install.packages(new_packages, dependencies = TRUE, quiet = TRUE)
invisible(lapply(required_packages, library, character.only = TRUE))
set.seed(42)


In [ ]:
# Rebuild the data, the split, the undersampled training set and the models exactly as in the main notebook
banking_url <- "https://raw.githubusercontent.com/umatter/EDFB/main/data/banking.csv"
dataset <- read_csv(banking_url, show_col_types = FALSE) %>%
  select(-duration, -pdays, -age, -campaign, -previous)
num_var <- names(select_if(dataset %>% select(-y), is.numeric))
cat_var <- names(select_if(dataset %>% select(-y), is.character))
X_raw <- dataset %>% select(-y) %>% mutate(across(all_of(cat_var), as.factor))
dummies <- dummyVars(~ ., data = X_raw, fullRank = TRUE, sep = "_")
dataset_dummy <- as.data.frame(predict(dummies, newdata = X_raw))
dataset_dummy$y <- dataset$y
col_to_drop <- c("emp_var_rate", "cons_price_idx", "euribor3m", "nr_employed", "loan_unknown")
dataset <- dataset_dummy %>% select(-all_of(col_to_drop))
num_var <- intersect(num_var, names(dataset))
X <- dataset %>% select(-y)
y <- dataset$y
set.seed(0)
train_indices <- createDataPartition(factor(dataset$y), p = 0.8, list = FALSE)
X_train <- X[train_indices, ]
X_test <- X[-train_indices, ]
y_train <- y[train_indices]
y_test <- y[-train_indices]
train_means <- colMeans(X_train[num_var])   # scaler fit on the training rows only
train_sds <- sapply(X_train[num_var], sd)
X_train[num_var] <- as.data.frame(scale(X_train[num_var], center = train_means, scale = train_sds))   # as.data.frame() keeps plain numeric columns
X_test[num_var] <- as.data.frame(scale(X_test[num_var], center = train_means, scale = train_sds))

# Undersample the training set only
train_data <- X_train
train_data$y <- y_train
train_1 <- train_data %>% filter(y == 1)
train_0 <- train_data %>% filter(y == 0)
set.seed(0)
train_0_small <- train_0 %>% slice_sample(n = 2 * nrow(train_1))
train_bal <- bind_rows(train_1, train_0_small) %>% slice_sample(prop = 1)
X_train_bal <- train_bal %>% select(-y)
y_train_bal <- train_bal$y
sampling_ratio <- nrow(train_0_small) / nrow(train_0)

# Models
lpm_model <- lm(y ~ ., data = train_bal)
y_test_pred_lpm <- predict(lpm_model, X_test)
logit_model <- glm(y ~ ., data = train_bal, family = binomial)
y_test_predicted_prob_logit <- predict(logit_model, X_test, type = "response")
p_test <- plogis(predict(logit_model, X_test, type = "link") + log(sampling_ratio))   # deployment probabilities

odds_table <- data.frame(feature = names(coef(logit_model)),
                         log_odds_coef = unname(coef(logit_model)),
                         odds_ratio = unname(exp(coef(logit_model)))) %>%
  filter(feature != "(Intercept)") %>%
  arrange(desc(abs(log_odds_coef)))

cat(sprintf("Test set: %d clients, share subscribed %.3f\n", length(y_test), mean(y_test)))
cat(sprintf("Sampling ratio %.3f; AUC on the test set %.3f\n", sampling_ratio,
            auc(roc(y_test, p_test, levels = c(0, 1), direction = "<", quiet = TRUE))))


## Exercise 1: LPM versus logit predictions

**Task:** Scatter the LPM probabilities against the logit probabilities on the test set with the 45-degree line; report the share of clients on which the two models agree at threshold 0.5, the largest absolute difference, and where along the probability range the models disagree.


In [ ]:
ggplot(data.frame(lpm = y_test_pred_lpm, logit = y_test_predicted_prob_logit), aes(x = lpm, y = logit)) +
  geom_point(alpha = 0.3, size = 0.8) +
  geom_abline(intercept = 0, slope = 1, color = "red", linetype = "dashed") +
  labs(title = "LPM versus logit on the test set", x = "LPM predicted probability", y = "Logit predicted probability") +
  theme_minimal()

lpm_class <- ifelse(y_test_pred_lpm >= 0.5, 1, 0)
logit_class <- ifelse(y_test_predicted_prob_logit >= 0.5, 1, 0)
agreement <- mean(lpm_class == logit_class)
diff <- abs(y_test_pred_lpm - y_test_predicted_prob_logit)
i_max <- which.max(diff)
cat(sprintf("Same 0/1 classification at 0.5 for %.1f%% of the test clients\n", 100 * agreement))
cat(sprintf("Largest disagreement: LPM %.3f vs logit %.3f (difference %.3f)\n",
            y_test_pred_lpm[i_max], y_test_predicted_prob_logit[i_max], diff[i_max]))

# Where do they disagree? Mean absolute difference by range of the logit probability
data.frame(logit_prob = y_test_predicted_prob_logit, abs_diff = diff) %>%
  mutate(bin = cut(logit_prob, c(0, 0.2, 0.4, 0.6, 0.8, 1))) %>%
  group_by(bin) %>%
  summarise(mean_abs_diff = mean(abs_diff), count = n(), .groups = "drop")


**Result.** In this run the two models make the same 0/1 classification for 99.7% of the test clients. The largest disagreement is a client with LPM value 1.004 and logit probability 0.934 (difference 0.070): the LPM runs above 1 where the logit flattens out. The mean absolute difference is below 0.01 between 0 and 0.4, where most clients sit, and largest (0.02 to 0.04) above 0.6, where the S-curve bends and the straight line cannot follow. For classification at 0.5 the choice of model barely matters; for probabilities near the boundaries it does.

## Exercise 2: Threshold sweep

**Task:** For thresholds 0.01 to 0.50 on the deployment probabilities `p_test`, compute the number of calls, precision and recall; plot them; show the rows for 0.05, 0.10 and 0.20 and describe the trade-off.


In [ ]:
thresholds <- round(seq(0.01, 0.50, by = 0.01), 2)   # round() so that 0.10 is exactly 0.10
sweep <- bind_rows(lapply(thresholds, function(t) {
  call <- p_test >= t
  tp <- sum(call & y_test == 1)
  data.frame(threshold = t, calls = sum(call),
             precision = ifelse(sum(call) > 0, tp / sum(call), NA),
             recall = tp / sum(y_test == 1))
}))

p_rates <- sweep %>%
  pivot_longer(c(precision, recall), names_to = "metric", values_to = "value") %>%
  ggplot(aes(x = threshold, y = value, color = metric)) +
  geom_line(linewidth = 1) +
  labs(title = "Threshold sweep on the test set", x = "threshold on p_test", y = "precision / recall") +
  theme_minimal()
p_calls <- ggplot(sweep, aes(x = threshold, y = calls)) +
  geom_line(color = "grey40", linewidth = 1) +
  labs(x = "threshold on p_test", y = sprintf("number of calls (of %d test clients)", length(y_test))) +
  theme_minimal()
options(repr.plot.width = 12, repr.plot.height = 5)
grid.arrange(p_rates, p_calls, ncol = 2)

sweep %>% filter(threshold %in% c(0.05, 0.10, 0.20))


**Result.** Lowering the threshold trades precision for recall through the number of calls. In this run: at 0.05 the bank calls 6,617 of the 8,237 test clients (80%), reaches 93% of the subscribers, and 13% of the calls end in a sale; at 0.10 it calls 4,197 (51%), reaches 72%, and precision is 16%; at 0.20 it calls 344 (4%), reaches 22%, and 59% of the calls succeed. The model ranks clients (the AUC is about 0.70), but few clients have a high probability, so precision only rises once the call list becomes very short.

## Exercise 3: Expected profit and the optimal threshold

**Task:** With a €5 call cost and €100 revenue per subscription, compute the profit per 10,000 clients for every threshold, compare with calling everyone, derive the rule `p > cost / revenue`, and repeat with a €20 call cost.


In [ ]:
contact_cost <- 5
subscription_revenue <- 100
total_customers <- 10000
n_test <- length(y_test)
thresholds <- round(seq(0.01, 0.50, by = 0.01), 2)

profit_per_10k <- function(threshold, cost, revenue = subscription_revenue) {
  call <- p_test >= threshold
  tp <- sum(call & y_test == 1)
  (revenue * tp - cost * sum(call)) / n_test * total_customers
}

profit_curves <- list()
for (cost in c(contact_cost, 20)) {
  profits <- sapply(thresholds, profit_per_10k, cost = cost)
  everyone <- (subscription_revenue * sum(y_test == 1) - cost * n_test) / n_test * total_customers
  best_t <- thresholds[which.max(profits)]
  rule_t <- cost / subscription_revenue
  cat(sprintf("Call cost EUR %d: break-even p = cost / revenue = %.2f\n", cost, rule_t))
  cat(sprintf("  call everyone (no model):     EUR %9.0f per 10,000 clients\n", everyone))
  cat(sprintf("  best threshold %.2f:          EUR %9.0f per 10,000 clients, calling %.0f%% of clients\n",
              best_t, max(profits), 100 * mean(p_test >= best_t)))
  cat(sprintf("  rule threshold %.2f:          EUR %9.0f per 10,000 clients, calling %.0f%% of clients\n",
              rule_t, profit_per_10k(rule_t, cost), 100 * mean(p_test >= rule_t)))
  profit_curves[[length(profit_curves) + 1]] <- data.frame(threshold = thresholds, profit = profits,
                                                           cost = paste("call cost EUR", cost))
}

options(repr.plot.width = 8, repr.plot.height = 5)
ggplot(bind_rows(profit_curves), aes(x = threshold, y = profit, color = cost)) +
  geom_line(linewidth = 1) +
  geom_hline(yintercept = 0, color = "grey50", linewidth = 0.3) +
  labs(title = "Expected profit by threshold", x = "threshold on p_test", y = "profit per 10,000 clients (EUR)", color = "") +
  theme_minimal()


**Result.** Call cost EUR 5: the break-even probability is 5 / 100 = 0.05. Calling everyone earns about EUR 62,700 per 10,000 clients; the best threshold (0.06 in this run) earns about EUR 64,900, and the rule threshold 0.05 about EUR 64,200. The model adds little (about EUR 2,300 per 10,000 clients at best, under 4%), and that is the honest result: with a base rate of 11.3% and a break-even of 5%, nearly every client is worth a call, so the decision hardly depends on the ranking. Call cost EUR 20: the break-even is 0.20. Calling everyone now loses about EUR 87,000 per 10,000 clients, while the rule threshold 0.20 earns about EUR 16,200 (calling 4% of clients) and the best threshold (0.17 in this run) about EUR 17,800. The rule in one sentence: call a client when the expected revenue p x EUR 100 exceeds the cost of the call, that is when p > cost / revenue. The empirical optimum lies close to that value; the flat profit curve around it is sampling noise on a finite test set. The rule needs the deployment probabilities `p_test`: the probabilities of the undersampled fit are inflated (their mean is 0.30 instead of 0.11), so applying p > 0.05 to them would call everyone and p > 0.20 would still call far too many. The value of the model depends on the economics of the campaign, not on the AUC alone.

## Exercise 4: Odds ratios in business terms

**Task:** Extract the odds ratios of `poutcome_success`, `contact_telephone` and `cons_conf_idx`, explain each in one sentence, and sort the model's features into what the manager controls, what she can target on, and what she cannot influence.


In [ ]:
three <- odds_table %>% filter(feature %in% c("poutcome_success", "contact_telephone", "cons_conf_idx"))
print(three, digits = 3, row.names = FALSE)

feature_names <- names(X_train)
groups <- list(
  "controls directly" = "contact_telephone",
  "can target on, cannot change" = feature_names[grepl("^(marital|education|housing|loan|poutcome)_", feature_names)],
  "cannot influence (macro)" = "cons_conf_idx"
)
for (g in names(groups)) cat(g, ":", paste(groups[[g]], collapse = ", "), "\n")


**Result.** `poutcome_success`, odds ratio about 10.1 in this run: a client who subscribed in the previous campaign has about 10 times the odds of subscribing again, other things equal (the unit is having the characteristic). `contact_telephone`, odds ratio about 0.38: a client reached on a landline has 38% of the odds of a client reached on a mobile phone, so about 60% lower odds. `cons_conf_idx`, odds ratio about 1.15 per standard deviation of consumer confidence (about 4.6 index points): one standard deviation more confidence raises the odds of a sale by about 15%. Groups: (a) the manager controls the contact channel (`contact_telephone`), with the caveat that the coefficient may partly reflect who is reachable by landline rather than the channel itself; (b) marital status, education, housing and personal loans and the outcome of the previous campaign are client characteristics she can use to decide whom to call but cannot change; (c) consumer confidence is a macro variable she cannot influence, except by timing the campaign.

## Exercise 5: Three clients

**Task:** Encode clients A, B and C with the columns of `X_train`, compute their deployment probabilities, rank them, and apply the €5/€100 rule (call if p > 0.05).


In [ ]:
customers <- as.data.frame(matrix(0, nrow = 3, ncol = ncol(X_train),
                                  dimnames = list(c("A", "B", "C"), names(X_train))))
# A: married, university degree, housing loan, no personal loan, cellular, previous campaign success, confidence average
customers["A", c("marital_married", "education_university.degree", "housing_yes", "poutcome_success")] <- 1
# B: single, high school, no loans, landline, never contacted before, confidence one SD below average
customers["B", c("marital_single", "education_high.school", "contact_telephone", "poutcome_nonexistent")] <- 1
customers["B", "cons_conf_idx"] <- -1
# C: divorced (reference), basic.9y, housing loan and personal loan, cellular (reference), previous failure (reference), confidence average
customers["C", c("education_basic.9y", "housing_yes", "loan_yes")] <- 1

logodds <- predict(logit_model, customers, type = "link") + log(sampling_ratio)
result <- data.frame(client = rownames(customers),
                     p_undersampled_fit = predict(logit_model, customers, type = "response"),
                     p_deploy = plogis(logodds))
result$call_at_5_over_100 <- result$p_deploy > 0.05
result %>% arrange(desc(p_deploy))


**Result.** Deployment probabilities in this run: A 0.646, C 0.112, B 0.049. Client A, with a previous success, is far above the 5% break-even and is called first. Client C sits near the base rate (11%) and is worth a call at EUR 5 per call. Client B is right at the break-even (0.049 here, so `call_at_5_over_100` is FALSE; the Python twin, with its different random split, puts the same client at 0.051, just above it): the expected profit of the call is about zero, so at EUR 5 the decision is a coin toss and at EUR 20 it is clearly no. The column `p_undersampled_fit` shows why the correction matters: on the uncorrected scale B looks like a 17% prospect and C like a 33% prospect.